In [69]:
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torch
import numpy as np
import cv2
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import pandas as pd
import torch.optim as optim
import segmentation_models_pytorch as smp

![U-Net Architecture](./U-net_Arch.png)

In [70]:
def RLE_to_mask(rle,shape=(256, 1600)):
    s = rle.split()
    starts, lengths = [np.asarray(x, dtype=int) for x in (s[0::2], s[1::2])]
    starts -= 1
    ends = starts + lengths

    img = np.zeros(shape[0]*shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1

    return img.reshape(shape, order='F')

In [71]:
class SteelDataset(Dataset):
    def __init__(self, df, img_folder, transform=None):
        self.df = df
        self.img_folder = img_folder
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = f"{self.img_folder}/{row['ImageId']}"
        
        # Load Image
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Decode Mask using your function
        mask = RLE_to_mask(row['EncodedPixels'], shape=(256, 1600))
        
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        if isinstance(image, np.ndarray):
            image = torch.from_numpy(image).permute(2, 0, 1).float()
        if isinstance(mask, np.ndarray):
            mask = torch.from_numpy(mask).float()

        # U-Net needs (Channels, H, W). Mask needs a channel dim: (1, H, W)
        return image, mask.unsqueeze(0).float()

In [72]:
df = pd.read_csv("data/kaggle_data/train.csv")
df.columns = ["ImageId", "ClassId", "EncodedPixels"]

steel_data = SteelDataset(df,"data/kaggle_data/train_images/")
train_loader = DataLoader(steel_data, batch_size=4, shuffle=True)
u_net = smp.Unet(
    encoder_name="resnet34",        # Choose a backbone (ResNet is great for steel)
    encoder_weights="imagenet",     # Use pre-trained weights to start
    in_channels=1,                  # 1 for grayscale steel images, 3 for RGB
    classes=4,                      # 1 for binary detection (defect vs no defect)
    activation='softmax2d'            # Output 0 to 1 probability
)

optimizer = optim.AdamW(u_net.parameters(), lr=1e-4, weight_decay=1e-2)
loss_fn = smp.losses.DiceLoss(mode='multiclass')

In [73]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [74]:
def train_fn(epochs, train_loader, net, optimizer):
    net.to(device)
    for epoch in range(epochs):
        net.train()
        epoch_loss = 0
        
        for images, targets in train_loader:
            # Move data to GPU
            images = list(image.to(device) for image in images)
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            # Forward pass: Returns a dictionary of losses
            loss_dict = net(images, targets)
            losses = sum(loss for loss in loss_dict.values())

            optimizer.zero_grad()
            losses.backward()
            optimizer.step()
            
            epoch_loss += losses.item()

        print(f"Epoch {epoch} | Avg Loss: {epoch_loss/len(train_loader):.4f}")

In [76]:
train_fn(70, train_loader, u_net, optimizer)

AttributeError: 'Tensor' object has no attribute 'items'